# Pipeline Architecture — Google ADK

**Pattern:** ETL via sequential tool calls

```
extract_records() → enrich_and_rank() → LLM formats report
```

ADK models pipeline steps as tools. The instruction enforces sequential execution.

In [ ]:
import asyncio, os, json
from dotenv import load_dotenv
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner
from google.adk.sessions import InMemorySessionService
from google.genai import types as genai_types

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY in .env"
RAW = ["Tokyo|Clear|18|Low|10|22:30 JST", "Paris|Partly Cloudy|16|Low|8|15:30 CET", "Bangalore|Rainy|25|Medium|6|20:00 IST"]
print("✓ Ready")

In [ ]:
def extract_records(raw_data: str) -> dict:
    """Parse pipe-delimited city records.
    Args:
        raw_data: Newline-separated pipe-delimited records.
    Returns:
        Dict with parsed records.
    """
    records = []
    for line in raw_data.strip().split("\n"):
        p = line.split("|")
        if len(p)==6:
            c,w,t,sl,ss,tm = p
            records.append({"city":c,"weather":w,"temp_c":int(t),"safety_level":sl,"safety_score":int(ss),"local_time":tm})
    return {"records": records, "count": len(records)}

def enrich_and_rank(json_records: str) -> dict:
    """Add scores and rank records.
    Args:
        json_records: JSON string of city records.
    Returns:
        Dict with enriched ranked records.
    """
    records = json.loads(json_records)
    sm = {"Clear":9,"Partly Cloudy":7,"Rainy":6,"Cloudy":5}
    for r in records:
        r["weather_score"] = sm.get(r["weather"],5)
        r["combined_score"] = round((r["weather_score"]+r["safety_score"])/2,1)
    records.sort(key=lambda x:x["combined_score"],reverse=True)
    for i,r in enumerate(records,1): r["rank"] = i
    return {"enriched_records": records}

agent = Agent(name="etl_pipeline", model="gemini-2.0-flash",
    description="ETL pipeline agent.",
    instruction="""ETL pipeline:\nSTEP 1: Call extract_records() with raw data.\nSTEP 2: Call enrich_and_rank() with extracted JSON.\nSTEP 3: Format enriched records into a markdown ranking report with a table.""",
    tools=[extract_records, enrich_and_rank])
print(f"Agent: {agent.name}")

In [ ]:
async def run(raw):
    ss = InMemorySessionService()
    session = await ss.create_session(app_name="etl_pipeline", user_id="u1")
    runner = InMemoryRunner(agent=agent, session_service=ss)
    final = ""
    async for event in runner.run_async(user_id=session.user_id, session_id=session.id,
        new_message=genai_types.Content(role="user", parts=[genai_types.Part(text=f"Run ETL:\n{'\\n'.join(raw)}")])):
        if hasattr(event,"tool_call") and event.tool_call: print(f"  [step] {event.tool_call.name}")
        elif event.is_final_response() and event.content:
            for p in event.content.parts:
                if p.text: final += p.text
    return final

report = await run(RAW)
print("\n--- Pipeline Report ---")
print(report)

## Key Takeaways

| What You Saw | Why It Matters |
|---|---|
| Tool calls for E + T steps | Pure deterministic transforms as tools |
| LLM only for L (formatting) | Same principle: LLM where narrative matters |
| STEP 1/2/3 in instruction | Pipeline order enforced via instruction |

### Pipeline: All 4 Frameworks
| Framework | Extract | Transform | Load |
|---|---|---|---|
| LangChain | Python function | Python function | LCEL chain |
| LangGraph | Graph node | Graph node | Graph node (LLM) |
| CrewAI | Agent + tool | Agent + tool | Agent (LLM) |
| ADK | Tool call | Tool call | LLM output |

### Next: [06-Adversarial-Debate →](../../06-Adversarial-Debate/LangChain/debate.ipynb)